Code to create vapour pressure deficit (VPD) projections from daily tasmax and hursmin projections.

Equations saturation vapour pressure http://www.bom.gov.au/climate/how/newproducts/images/IDCJHC02_notes.txt

Vapour pressure = exp (1.8096 + (17.269425 * Dew_Point)/(237.3 + Dew_Point))

Saturated Vapour pressure = exp (1.8096 + (17.269425 * Air_Temperature)/(237.3 + Air_Temperature))

Relative Humidity = Vapour pressure / Saturated vapour pressure * 100

Rearrange the formulae to get:

Vapour pressure = rh * 0.0061094 * exp((17.652 * t)/(243.04 + t))

A nice explainer about the relevance of VPD to fire: https://blog.ucsusa.org/carly-phillips/what-is-vapor-pressure-deficit-vpd-and-what-is-its-connection-to-wildfires/

In [1]:
import dask
from dask.distributed import Client, wait
from dask import delayed

client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43249,Workers: 7
Dashboard: /proxy/8787/status,Total threads: 14
Started: Just now,Total memory: 63.00 GiB
Comm: tcp://127.0.0.1:37961,Total threads: 2
Dashboard: /proxy/45951/status,Memory: 9.00 GiB
Nanny: tcp://127.0.0.1:35901,


2025-03-24 10:56:53,056 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/EC-Earth3/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/ssp370_EC-Earth3_BARPA-R_gwl3.0_vpd.nc', lease_id='f1f10cd013b54b3d803039252696fbf5'. This can happen if the Lock or Semaphore timed out before.


In [2]:
#import all the stuff
import xarray as xr
import numpy as np
import pandas as pd
from datetime import timedelta
import glob
import sys
sys.path.append("/g/data/mn51/users/nb6195/project/gwls/")
import gwl

In [3]:
#function to compute VPD from tasmax and rh datasets
#input: are datasets of rh and tasmax
#output: datasets of vpd and monthly_mean_vpd

def vpd_calc(ds_rh, ds_tasmax):
#    vpd = (1 - ds_rh/100) * 0.61094 * np.exp((17.652 * ds_tasmax)/(243.04 + ds_tasmax)) #originally used funtion
    vpd = (1 - ds_rh/100) * 6.1094 * np.exp ((17.625 * ds_tasmax)/(243.04 + ds_tasmax)) #the formula that Blair uses from http://www.bom.gov.au/research/publications/cawcrreports/CTR_024.pdf
    monthly_mean_vpd = vpd.groupby('time.month').mean('time', keep_attrs=True)
    
    vpd.attrs = {
        'long_name': 'Daily maximum vapour dressure deficit computed from tasmax and hursmin',
        'standard_name': 'vpd',
        'units': 'hPa',
#        'regrid_method': 'bilinear'
    }
    ds_vpd = xr.Dataset({'vpd' : vpd})

    monthly_mean_vpd.attrs = {
        'long_name': 'Monthly mean vapour dressure deficit computed from tasmax and hursmin',
        'standard_name': 'monthly_mean_vpd',
        'units': 'hPa',
    }
    ds_monthly_mean_vpd = xr.Dataset({'monthly_mean_vpd' : monthly_mean_vpd})
    return ds_vpd, ds_monthly_mean_vpd

BOM to do:
- CESM2
- CMCC-ESM2
- NorESM2-MM

BOM done: 
- ACCESS-CM2 (r4i1p1f1)
- ACCESS-ESM1-5 (r6i1p1f1)
- EC-Earth3 (r1i1p1f1)
- MPI-ESM1-2-HR (r1i1p1f1)

CSIRO to do:   
- CESM2
- CMCC-ESM2
- EC-Earth3
- NorESM2-MM

CSIRO done: 
- ACCESS-CM2 (r4i1p1f1)
- ACCESS-ESM1-5 (r6i1p1f1)
- CNRM-ESM2-1 (r1i1p1f2)

In [50]:
#Set parameters
CMIP='CMIP6'
#AGENCY = 'CSIRO' 
#RCM = 'CCAM-v2203-SN'
AGENCY = 'BOM' 
RCM = 'BARPA-R'

#GCM = 'ACCESS-ESM1-5' #ensemble = 'r6i1p1f1' #Done
#GCM = 'ACCESS-CM2' #ensemble = 'r4i1p1f1' #Done
#GCM = 'CNRM-ESM2-1' #ensemble = 'r1i1p1f2' #CSIRO Done, no BOM
#GCM = 'MPI-ESM1-2-HR' #ensemble = 'r1i1p1f1' #BOM done, no CSIRO


#GCM = 'EC-Earth3' #BOM done, CSIRO not working
#ensemble = 'r1i1p1f1'

#GCM = 'NorESM2-MM' #datetime issue
#ensemble = 'r1i1p1f1'

#GCM = 'CMCC-ESM2'
#ensemble = 'r1i1p1f1'

GCM = 'CESM2'
ensemble = 'r11i1p1f1'

#pathway = 'ssp126'
pathway = 'ssp370'

output_dir = '/g/data/ia39/ncra/bushfire/vpd/'
output_dir_mm = '/g/data/ia39/ncra/bushfire/vpd/monthly_mean/'

In [53]:
#read in RCM files
var1 = 'tasmax'

ddir = f"/g/data/ia39/australian-climate-service/release/CORDEX/output-Adjust/{CMIP}/bias-adjusted-input/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var1}/v20241216"
infiles1=glob.glob(ddir+f'/{var1}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
tasmax_master_ds = xr.open_mfdataset(infiles1)

var2 = 'hursmin'

ddir = f"/g/data/ia39/australian-climate-service/release/CORDEX/output-Adjust/{CMIP}/bias-adjusted-input/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var2}/v20241216"
infiles2=glob.glob(ddir+f'/{var2}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
hursmin_master_ds = xr.open_mfdataset(infiles2)

In [54]:
#Extract time period corresponding to the chosen GWL for tasmax and rh
chosen_gwl = '1.2'

gwl_tasmax = gwl.get_GWL_timeslice(tasmax_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var1]
gwl_rh = gwl.get_GWL_timeslice(hursmin_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var2]
gwl_rh = gwl_rh.assign_coords(time = pd.to_datetime(gwl_rh.time) + timedelta(hours = 12))

/jobfs/137475449.gadi-pbs/ipykernel_2364271/2834203322.py:6: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  gwl_rh.indexes['time'].to_datetimeindex()


DatetimeIndex(['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04',
               '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08',
               '2015-01-09', '2015-01-10',
               ...
               '2029-12-22', '2029-12-23', '2029-12-24', '2029-12-25',
               '2029-12-26', '2029-12-27', '2029-12-28', '2029-12-29',
               '2029-12-30', '2029-12-31'],
              dtype='datetime64[ns]', length=5475, freq=None)

In [ ]:
#gwl_rh.indexes['time'].to_datetimeindex()

#gwl_rh = gwl_rh.assign_coords(time = gwl_rh.indexes['time'].to_datetimeindex() + timedelta(hours = 12))
#gwl_rh['month'] = gwl_rh['time'].dt.month
#gwl_tasmax['month'] = gwl_tasmax['time'].dt.month

In [ ]:
gwl_rh.time.dt.calendar

In [ ]:
#Create the datasets for vpd and monthly mean vpd
gwl_vpd, monthly_mean_vpd = vpd_calc(gwl_rh, gwl_tasmax)

In [ ]:
#monthly_mean_vpd

In [ ]:
#print vpd to an external file

file_name_vpd = pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd.nc' 
output_file_location = output_dir + GCM + '/' + pathway + '/' + ensemble + '/' + RCM + '/v1-r1/day/' + file_name_vpd
gwl_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

In [ ]:
#print monthly mean ds to external file

file_name_mean = 'gwl' + chosen_gwl + '_monthly_mean_vpd_' + GCM + '_' + RCM + '_' + pathway + '_' + ensemble + '.nc' 
output_file_location = output_dir_mm + file_name_mean
monthly_mean_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)